# Final Corpus Classification — Gemini 3.1 Pro (v2)

Manual batch classification workflow. Since the Gemini API is not available, this notebook:
1. **Generates the prompt** to copy-paste into Gemini.
2. **Provides a cell** to paste Gemini's response.
3. **Parses and evaluates** the results automatically.

> **Note:** The standard / irony / obfuscated splits share identical test sets within each corpus type.

## Dependencies

In [ ]:
import os
import re

import pandas as pd

from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

import watermark

%load_ext watermark
%matplotlib inline

plt.style.use('../style.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
%watermark -n -v -m -iv

## Prompt builder

In [ ]:
SYSTEM_MSG = "Sos un clasificador binario de tweets que hablan sobre drogas ilícitas"

CRITERIA = """\
    Clasifica cada tweet como POSITIVE o NEGATIVE según estos criterios:

    POSITIVE: cumple con uno o más de los siguientes:
    - El usuario del tweet habla de cómo o qué tipo de droga ilícita está consumiendo.
    - El usuario del tweet expresa la necesidad de consumir drogas ilícitas, ya sea por abstinencia o por gusto.
    - El usuario añora consumir drogas ilícitas.

    NEGATIVE: no cumple con ningún criterio POSITIVE, por ejemplo:
    - Habla sobre noticias o información general sobre drogas ilícitas.
    - Menciona drogas ilícitas sin relación con consumo problemático o necesidad.
    - Expresa ironía o sarcasmo relacionado con drogas ilícitas.

    Tener en cuenta los siguientes aspectos:
    - En el tweet puede estar presente la ironía o sarcasmo.
    - El análisis se centra en el autor del tweet. Por ejemplo, si el tweet cita a otro usuario y le pregunta si ha consumido drogas, o si habla en nombre de otro usuario mencionando que ese usuario consume drogas, la clasificación del tweet debe ser NEGATIVE, ya que el contenido no involucra directamente al autor.
    - Algunos tweets mencionan tomar una línea de colectivo, subte o tren, pero solamente esto, no es condición suficiente para interpretarlo como una referencia al consumo de drogas ilícitas."""

def build_batch_prompt(tweets: list[str]) -> str:
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(tweets))
    return (
        f"[System instruction: {SYSTEM_MSG}]\n\n"
        + CRITERIA
        + "\n\n"
        + "Responde con una lista numerada, exactamente una clasificación por línea, "
        + "únicamente la palabra POSITIVE o NEGATIVE (sin puntuación adicional ni explicaciones):\n\n"
        + numbered
    )

def parse_batch_response(response_text: str, n: int) -> list[str]:
    lines = [l.strip() for l in response_text.strip().splitlines() if l.strip()]
    results = []
    for line in lines:
        cleaned = re.sub(r"^\d+[.)\s]+", "", line).strip().upper().rstrip(".")
        if cleaned in ("POSITIVE", "NEGATIVE"):
            results.append(cleaned)
    if len(results) != n:
        raise ValueError(f"Expected {n} classifications, got {len(results)}. Response:\n{response_text}")
    return results

## Corpus types

In [ ]:
CORPORA = [
    ("raw-corpus",           "../data/raw/final-corpus/raw-corpus/standard/test.csv"),
    ("pre-filtered-corpus",  "../data/raw/final-corpus/pre-filtered-corpus/standard/test.csv"),
]

## Step 1 — Generate prompts to paste into Gemini

Run this cell, then copy each prompt into [Gemini](https://gemini.google.com/) and paste the response in **Step 2**.

In [ ]:
dfs = {}
prompts = {}

for corpus_label, csv_path in CORPORA:
    df = pd.read_csv(csv_path)
    dfs[corpus_label] = df
    tweets = df["text"].tolist()
    prompt = build_batch_prompt(tweets)
    prompts[corpus_label] = prompt

    print(f"\n{'='*70}")
    print(f"PROMPT FOR: {corpus_label}  ({len(tweets)} tweets)")
    print(f"{'='*70}")
    print(prompt)
    print(f"\n{'='*70}\n")

## Step 2 — Paste Gemini responses here

After running each prompt in Gemini, paste the full response into the corresponding variable below.

> Leave the string empty (`""`) for corpora you haven't run yet.

In [ ]:
# Paste Gemini's response for each corpus below:

GEMINI_RESPONSES = {
    "raw-corpus": """# PASTE raw-corpus response here
""",
    "pre-filtered-corpus": """# PASTE pre-filtered-corpus response here
""",
}

## Step 3 — Parse responses and compute metrics

In [ ]:
LABELS = ["POSITIVE", "NEGATIVE"]
results = {}

for corpus_label, _ in CORPORA:
    response_text = GEMINI_RESPONSES[corpus_label]
    if not response_text.strip() or response_text.strip().startswith("# PASTE"):
        print(f"Skipping {corpus_label} — no response pasted yet.")
        continue

    df = dfs[corpus_label].copy()
    try:
        predictions = parse_batch_response(response_text, len(df))
    except ValueError as e:
        print(f"Parse error for {corpus_label}: {e}")
        continue

    df["prediction"] = predictions
    results[corpus_label] = df

    y_true = df["label"]
    y_pred = df["prediction"]

    print(f"\n{'='*60}")
    print(f"Corpus: {corpus_label}")
    print(f"{'='*60}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(2, 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS,
                cbar=True, annot_kws={"size": 9}, ax=ax)
    ax.collections[0].colorbar.ax.tick_params(labelsize=6)
    ax.set_xlabel("Prediction", fontsize=6)
    ax.set_ylabel("True", fontsize=6)
    ax.tick_params(labelsize=5)
    ax.set_title(f"Confusion Matrix — {corpus_label}", fontsize=9)
    plt.tight_layout()
    plt.show()
    print(cm)

    # Classification report
    report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    print("\nClassification report:")
    display(report_df.style.format({"precision": "{:.2f}", "recall": "{:.2f}", "f1-score": "{:.2f}", "support": "{:.0f}"}))

    # Errors
    df["error"] = df["label"] != df["prediction"]
    errors = df[df["error"]][["text", "label", "prediction"]]
    print(f"\nWrongly classified: {len(errors)}")
    display(errors)

## Step 4 — Save wrongly-classified tweets

In [ ]:
for corpus_label, _ in CORPORA:
    if corpus_label not in results:
        continue
    df = results[corpus_label]
    errors = df[df["label"] != df["prediction"]][["text", "label", "prediction"]]
    out_path = f"../data/processed/final-corpus/{corpus_label}/wct_gemini-3.1-pro-v2.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    errors.to_csv(out_path, index=False)
    print(f"Saved {len(errors)} errors → {out_path}")